# PROMPT ENGINEERING AND RAG
## SESSION 3

Assignment questions and answers


## Question 1

Mini Project: Food Delivery Help Desk Chatbot
(RAG-Powered Console App)

Build a console-based help-desk chatbot for a food-delivery platform
combining document chunking, FAISS-based retrieval, structured prompt
engineering, and few-shot complaint classification in one interactive
program with session logging.

Requirements: - Provide a menu with at least three options: (1) Ask a
policy question via RAG, (2) Classify a complaint using few-shot
prompting, and (3) Exit. Continue showing the menu until Exit is
selected. - For option 1, load and chunk a predefined food-delivery
policy text of at least 500 words covering refunds, cancellations, and
delivery rules at startup. Embed all chunks using sentence-transformers
and build a FAISS index. For each query, retrieve the top three relevant
chunks, assemble a structured RAG prompt, and print retrieved context
and the complete prompt. - For option 2, maintain a few-shot example
bank with at least four labelled complaints. Accept a user complaint,
build and print the few-shot classification prompt, and display the
predicted category along with the closest matching example from the
bank. - Validate user input: reject empty questions and blank complaint
text, display a user-friendly error, and re-prompt without crashing. -
Maintain a plain-text session log recording each interaction’s
timestamp, mode (RAG / Classify), query text, and output. On exit, print
the total policy questions asked and complaints classified during the
session.

**Answer:**

```python
from datetime import datetime
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Initialize Model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Knowledge Base (>500 words) & FAISS Setup
POLICY_TEXT = """
Food Delivery Platform Terms, Refund Rules, Cancellations, and Delivery Policies.
Section 1: Order Cancellation Rules. Customers may cancel an order free of charge within sixty seconds of placement. Once the restaurant accepts the order, cancellation incurs a fifty percent fee. If the kitchen begins food preparation, cancellation incurs an eighty percent penalty. After a delivery partner is assigned and en route to pick up the meal, orders cannot be cancelled, and no refund will be issued. The platform reserves the right to cancel orders if the restaurant is closed or items are out of stock. In platform-initiated cancellations, a one hundred percent refund is processed automatically.

Section 2: Refund and Return Guidelines. Refunds are evaluated on a case-by-case basis. If an entire order is missing, customers must report within two hours to receive a full refund or re-delivery credit. For missing individual items, photographic proof of the delivery receipt and the delivered bag is required within twenty-four hours. Refunds for spoiled, undercooked, or contaminated meals require high-resolution photographs submitted via the support portal. Tampered packaging must be reported immediately upon delivery. Refunds are credited to the original payment method within five to seven business days or instantly as platform wallet credits. Wallet credits expire after twelve months.

Section 3: Delivery Protocols and Delays. Standard delivery windows range from thirty to forty-five minutes based on distance, traffic, and weather. If delivery exceeds the estimated time by more than thirty minutes, the customer receives a delay voucher worth twenty percent of the order value. Couriers will wait at the delivery location for a maximum of eight minutes. During this period, the driver will call the customer three times. If the customer is unreachable after eight minutes, the order is cancelled with no refund, and the food is discarded.

Section 4: Address Changes and Delivery Instructions. Delivery addresses can only be modified before the restaurant starts preparing the order. Any change requested after preparation incurs an additional distance surcharge. Couriers are not obligated to enter multi-story private residences beyond common lobby areas unless contact-free delivery is explicitly selected. Contact-free deliveries require couriers to take a photograph of the dropped package at the doorstep. The platform is not liable for stolen packages once the drop-off photo confirmation is uploaded.

Section 5: Damaged Goods and Quality Assurance. In the event of spilled beverages or crushed containers, customers must provide clear images showing the damaged seal and item within one hour. Compensation will be provided as a pro-rated refund covering the damaged portions. Taste preference complaints are not eligible for refunds. Temperature-related complaints are only valid if delivery exceeded sixty minutes.
"""


def chunk_text(text, chunk_size=60, overlap=15):
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = " ".join(words[i : i + chunk_size])
        chunks.append(chunk)
        if i + chunk_size >= len(words):
            break
    return chunks


chunks = chunk_text(POLICY_TEXT)
chunk_embeddings = embedder.encode(chunks)
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype("float32"))

# 3. Few-Shot Complaint Data
FEW_SHOT_BANK = [
    {
        "text": "The pizza arrived completely cold and the box was crushed.",
        "label": "Damaged Food",
    },
    {
        "text": "I ordered two burgers but only received one in the bag.",
        "label": "Missing Items",
    },
    {
        "text": "The delivery driver was 45 minutes late and very rude.",
        "label": "Delivery Issue",
    },
    {
        "text": "I was charged twice on my credit card for the same lunch order.",
        "label": "Billing/Refund",
    },
]
example_embeddings = embedder.encode([ex["text"] for ex in FEW_SHOT_BANK])

# 4. State & Logging
stats = {"RAG": 0, "Classify": 0}


def log_session(mode, query, output):
    with open("session_log.txt", "a", encoding="utf-8") as f:
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(
            f"[{timestamp}] MODE: {mode}\nQUERY: {query}\nOUTPUT:\n{output}\n{'-'*50}\n"
        )


def get_non_empty_input(prompt_text):
    while True:
        val = input(prompt_text).strip()
        if val:
            return val
        print("Error: Input cannot be empty. Please try again.")


# 5. Main Console Loop
while True:
    print("\n=== Food Delivery Help Desk ===")
    print("1. Ask a policy question (RAG)")
    print("2. Classify a complaint (Few-Shot)")
    print("3. Exit")
    choice = input("Select an option (1-3): ").strip()

    if choice == "1":
        query = get_non_empty_input("\nEnter your policy question: ")
        q_emb = embedder.encode([query]).astype("float32")
        _, indices = index.search(q_emb, k=3)

        retrieved_context = "\n---\n".join([chunks[i] for i in indices[0]])
        rag_prompt = (
            f"System: You are a support agent. Answer using the context below.\n"
            f"Context:\n{retrieved_context}\n\n"
            f"User Question: {query}\n"
            f"Answer:"
        )

        output_display = (
            f"\n--- Retrieved Context ---\n{retrieved_context}\n"
            f"\n--- Assembled RAG Prompt ---\n{rag_prompt}\n"
        )
        print(output_display)

        stats["RAG"] += 1
        log_session("RAG", query, output_display)

    elif choice == "2":
        complaint = get_non_empty_input("\nEnter customer complaint text: ")

        # Find closest example via cosine distance / L2
        c_emb = embedder.encode([complaint]).astype("float32")
        dists = np.linalg.norm(example_embeddings - c_emb, axis=1)
        closest_idx = np.argmin(dists)
        closest_example = FEW_SHOT_BANK[closest_idx]

        prompt_examples = "\n".join(
            [f"Complaint: {ex['text']}\nCategory: {ex['label']}" for ex in FEW_SHOT_BANK]
        )
        classification_prompt = (
            f"Classify the complaint into one of the categories: Damaged Food, Missing Items, Delivery Issue, Billing/Refund.\n\n"
            f"{prompt_examples}\n"
            f"Complaint: {complaint}\nCategory:"
        )

        predicted_category = closest_example["label"]
        output_display = (
            f"\n--- Few-Shot Classification Prompt ---\n{classification_prompt}\n\n"
            f"Closest Matching Example: \"{closest_example['text']}\" -> {closest_example['label']}\n"
            f"Predicted Category: {predicted_category}\n"
        )
        print(output_display)

        stats["Classify"] += 1
        log_session("Classify", complaint, output_display)

    elif choice == "3":
        print("\nExiting Help Desk.")
        print(f"Total Policy Questions (RAG): {stats['RAG']}")
        print(f"Total Complaints Classified: {stats['Classify']}")
        break

    else:
        print("Invalid selection. Please enter 1, 2, or 3.")
```